# Validation:
BOS → predicts A

BOS A → predicts B

BOS A B → predicts C

BOS A B C → predicts D

Where the input is always the true sequence.
# Free generation
BOS → predicts A

BOS A → predicts B

BOS A B → predicts X   ← mistake!!

BOS A B X → predicts Y

BOS A B X Y → predicts Z

At this point it is operating in a distribution it did not see during training. That's the type of behavior we want to diagnose.

Let's look at the different questions the following functions respond to:

- `validate()`: "How good is my next-token prediction?"

- `teacher_forced_reconstruction()`: "What image do my next-token predictions corresponds to?"

- `generate_and_visualize_textures()`: "What happens when my model generates a complete sequence on its own?"

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
import matplotlib.pyplot as plt
import time
import numpy as np
import math
import sys

In [3]:
REPO_DIR = '/content/Texture_synthesis'
os.chdir('/content')

if not os.path.exists(REPO_DIR):
  !git clone https://github.com/ValentinaEmili/Texture-synthesis.git Texture_synthesis

if REPO_DIR not in sys.path:
  sys.path.append(REPO_DIR)

Cloning into 'Texture_synthesis'...
remote: Enumerating objects: 614, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (162/162), done.
remote: Total 614 (delta 94), reused 8 (delta 8), pack-reused 437 (from 1)
Receiving objects: 100% (614/614), 78.41 MiB | 24.77 MiB/s, done.
Resolving deltas: 100% (283/283), done.


In [4]:
!pip install import-ipynb -q
import import_ipynb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 49.1 MB/s eta 0:00:00


In [ ]:
from Texture_synthesis.codebook.VQ_VAE import VQ_VAE
from Texture_synthesis.generation.Transformers.Transformers import Transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# VQ-VAE
#quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae/best_1.pth'
#quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae/best_2.pth'
#quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae/best_3.pth'
quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_vae/best_1_2.pth'
quantizer = VQ_VAE().to(device)
checkpoint = torch.load(quant_save_path, map_location=torch.device('cpu'))
quantizer.load_state_dict(checkpoint['model_state_dict'])

transformer = Transformer(num_embeddings=512, seq_len=1024, n_layers=8).to(device)
optimizer = optim.Adam(transformer.parameters(), lr=2e-4, betas=(0.9, 0.999))

#best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_1/vqvae_best.pth'
#best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_2/vqvae_best.pth'
#best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_3/vqvae_best.pth'
best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_1_2/vqvae_best.pth'

In [ ]:
from Texture_synthesis.codebook.VQGAN.VQGAN import VQGAN, Discriminator
from Texture_synthesis.generation.Transformers.Transformers import Transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# VQ-GAN
quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_gan/best_generator_1/main_model.pth'
#quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_gan/gen_best_2.pth'
quantizer = VQGAN().to(device)
checkpoint = torch.load(quant_save_path)
quantizer.load_state_dict(checkpoint)
#quantizer.load_state_dict(checkpoint['model_state_dict'])

transformer = Transformer(num_embeddings=512, seq_len=1024, n_layers=8).to(device)
optimizer = optim.Adam(transformer.parameters(), lr=2e-4, betas=(0.9, 0.999))

best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_1/vqgan_best.pth'
#best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_2/vqgan_best.pth'

In [ ]:
train_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.RandomCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

In [7]:
from Texture_synthesis.codebook.VQGAN.VQGAN_TT import VQGAN, Discriminator
from Texture_synthesis.generation.Transformers.Transformers import Transformer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# VQ-GAN from Taming Transformers https://github.com/compvis/taming-transformers
quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_gan/taming_transformers/best_generator_1/main_model.pth'
#quant_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/vq_gan/taming_transformers/best_generator_2/main_model.pth'
quantizer = VQGAN().to(device)
checkpoint = torch.load(quant_save_path, map_location=torch.device('cpu'))
quantizer.load_state_dict(checkpoint)
#quantizer.load_state_dict(checkpoint['model_state_dict'])

transformer = Transformer(num_embeddings=1024, seq_len=256, n_layers=8).to(device)
optimizer = optim.Adam(transformer.parameters(), lr=2e-4, betas=(0.9, 0.999))

best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_1/vqgan_tt_best.pth'
#best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_2/vqgan_tt_best.pth'

In [8]:
# VQ-GAN from Taming Transformers paper

train_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.RandomCrop(256),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(256),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

In [9]:
class DTD_Dataset(Dataset):
    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [10]:
split = '1'
#split = '2'
#split = '_1_2'

train_file = 'train' + split + '.txt'
val_file = 'val' + split + '.txt'
test_file = 'test' + split + '.txt'

In [11]:
batch_size = 8
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, train_file), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, val_file), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, test_file), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [12]:
@torch.no_grad()
def extract_codebook_indices(quantizer, data):
  quantizer.eval()

  embedding_dim = quantizer.vq.embedding_dim
  embeddings = quantizer.vq.embeddings.weight                               # (num_embeddings, embed_dim)

  z = quantizer.encoder(data)                                               # (batch_size, embed_dim, h, w)
  z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, embedding_dim)  # (batch_size * h * w, embed_dim)

  distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(embeddings**2, dim=1)
                    - 2 * torch.matmul(z_flattened, embeddings.t()))        # (batch_size * h * w, num_embeddings)

  indices = torch.argmin(distances, dim=1)        # (batch_size * h * w)
  return indices.reshape(z.shape[0],-1)           # (batch_size, seq_len)

# Training and validation

In [ ]:
@torch.no_grad()
def validate(val_loader, transformer, quantizer, device):
  transformer.eval()
  quantizer.eval()

  transformer.to(device)
  quantizer.to(device)

  bos_token_id = transformer.num_embeddings

  total_loss, total_accuracy, total_tokens = 0.0, 0.0, 0.0

  for batch_idx, (data, _) in enumerate(val_loader):
    data = data.to(device)

    indices = extract_codebook_indices(quantizer, data)
    b, t = indices.shape                      # (batch_size, seq_len)

    bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
    full_sequence = torch.cat([bos_tokens, indices], dim=1) # (batch_size, seq_len)

    inputs = full_sequence[:, :-1]            # [BOS, c1, c2, ..., c_{t-1}]
    targets = full_sequence[:, 1:]            # [c1, c2, ..., c_{t-1}, c_{t}]

    num_tokens = targets.numel()
    total_tokens += num_tokens

    logits = transformer(inputs)              # (batch_size, seq_len, num_embeddings)

    predictions = torch.argmax(logits, dim=-1)
    accuracy = (predictions == targets).sum().item()

    targets = targets.reshape(-1)                             # (batch_size * seq_len)
    logits = logits.reshape(-1, transformer.num_embeddings) # (batch_size * seq_len, num_embeddings)

    loss = F.cross_entropy(logits, targets, reduction="sum")

    total_loss += loss.item()
    total_accuracy += accuracy

  mean_loss = total_loss / total_tokens
  mean_accuracy = (total_accuracy / total_tokens) * 100
  return mean_loss, mean_accuracy

In [ ]:
def train_and_validation(train_loader, val_loader, transformer, quantizer, optimizer, device, best_val_loss=float('inf'), epochs=20):
  transformer.to(device)
  quantizer.to(device)

  bos_token_id = transformer.num_embeddings

  for epoch in range(next_epoch, epochs):
    transformer.train()
    quantizer.eval()
    total_loss, total_accuracy, total_tokens = 0.0, 0.0, 0

    for batch_idx, (data, _) in enumerate(train_loader):
      data = data.to(device)

      indices = extract_codebook_indices(quantizer, data)
      b, t = indices.shape                      # (batch_size, seq_len)

      bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
      full_sequence = torch.cat([bos_tokens, indices], dim=1) # (batch_size, seq_len)

      inputs = full_sequence[:, :-1]            # [BOS, c1, c2, ..., c_{t-1}]
      targets = full_sequence[:, 1:]            # [c1, c2, ..., c_{t-1}, c_{t}]

      num_tokens = targets.numel()
      total_tokens += num_tokens

      logits = transformer(inputs)              # (batch_size, seq_len, num_embeddings)

      predictions = torch.argmax(logits, dim=-1)
      accuracy = (predictions == targets).sum().item()

      optimizer.zero_grad()

      targets = targets.reshape(-1)                             # (batch_size * seq_len)
      logits = logits.reshape(-1, transformer.num_embeddings) # (batch_size * seq_len, num_embeddings)

      loss = F.cross_entropy(logits, targets)

      loss.backward()
      optimizer.step()

      total_loss += loss.item() * num_tokens
      total_accuracy += accuracy

    train_loss = total_loss / total_tokens
    train_accuracy = (total_accuracy / total_tokens) * 100
    train_perplexity = np.exp(train_loss)

    print(f"====> Epoch {epoch} Finished")
    print(f"====> Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f} | Train Perplexity: {train_perplexity:.2f}")

    val_loss, val_accuracy = validate(val_loader, transformer, quantizer, device)
    val_perplexity = np.exp(val_loss)
    print(f"====> Valid Loss: {val_loss:.4f} | Valid Accuracy: {val_accuracy:.4f} | Valid Perplexity: {val_perplexity:.2f}")

    if val_loss < best_val_loss:
      best_val_loss = val_loss
      torch.save({
          'epoch': epoch,
          'model_state_dict':transformer.state_dict(),
          'optimizer_state_dict': optimizer.state_dict(),
          'best_val_loss': best_val_loss,
          'perplexity': val_perplexity,
          'accuracy': val_accuracy
          }, best_save_path)
      print(f"New best model at epoch {epoch}\n")

In [ ]:
train_and_validation(train_loader, val_loader, transformer, quantizer, optimizer, device, epochs=50)

====> Epoch 0 Finished
====> Train Loss: 3.4053 | Train Accuracy: 16.7558 | Train Perplexity: 30.12
====> Valid Loss: 2.9758 | Valid Accuracy: 20.4691 | Valid Perplexity: 19.61
New best model at epoch 0

====> Epoch 1 Finished
====> Train Loss: 2.9184 | Train Accuracy: 21.2265 | Train Perplexity: 18.51
====> Valid Loss: 2.8802 | Valid Accuracy: 21.4102 | Valid Perplexity: 17.82
New best model at epoch 1

====> Epoch 2 Finished
====> Train Loss: 2.8506 | Train Accuracy: 21.9739 | Train Perplexity: 17.30
====> Valid Loss: 2.8369 | Valid Accuracy: 21.9562 | Valid Perplexity: 17.06
New best model at epoch 2

====> Epoch 3 Finished
====> Train Loss: 2.8158 | Train Accuracy: 22.4462 | Train Perplexity: 16.71
====> Valid Loss: 2.8150 | Valid Accuracy: 22.2103 | Valid Perplexity: 16.69
New best model at epoch 3

====> Epoch 4 Finished
====> Train Loss: 2.7900 | Train Accuracy: 22.7421 | Train Perplexity: 16.28
====> Valid Loss: 2.7851 | Valid Accuracy: 22.4713 | Valid Perplexity: 16.20
New bes

In [ ]:
checkpoint = torch.load(best_save_path, map_location=torch.device("cpu"),weights_only=False)
transformer.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
next_epoch = checkpoint['epoch'] + 1
best_val_loss = checkpoint['best_val_loss']
train_and_validation(train_loader, val_loader, transformer, quantizer, optimizer, device, best_val_loss=best_val_loss, epochs=50)

====> Epoch 37 Finished
====> Train Loss: 2.3104 | Train Accuracy: 32.8720 | Train Perplexity: 10.08
====> Valid Loss: 2.3340 | Valid Accuracy: 32.1010 | Valid Perplexity: 10.32
New best model at epoch 37

====> Epoch 38 Finished
====> Train Loss: 2.3013 | Train Accuracy: 33.1169 | Train Perplexity: 9.99
====> Valid Loss: 2.3296 | Valid Accuracy: 32.2164 | Valid Perplexity: 10.27
New best model at epoch 38

====> Epoch 39 Finished
====> Train Loss: 2.2920 | Train Accuracy: 33.3145 | Train Perplexity: 9.90
====> Valid Loss: 2.3226 | Valid Accuracy: 32.4486 | Valid Perplexity: 10.20
New best model at epoch 39

====> Epoch 40 Finished
====> Train Loss: 2.2825 | Train Accuracy: 33.5262 | Train Perplexity: 9.80
====> Valid Loss: 2.3160 | Valid Accuracy: 32.5885 | Valid Perplexity: 10.13
New best model at epoch 40

====> Epoch 41 Finished
====> Train Loss: 2.2768 | Train Accuracy: 33.6801 | Train Perplexity: 9.75
====> Valid Loss: 2.3083 | Valid Accuracy: 32.7538 | Valid Perplexity: 10.06
Ne

## Results

### Split 1

#### VQ-VAE + Transformer

Train Loss: 2.3248 | Train Accuracy: 33.3592 | Train Perplexity: 10.22

Valid Loss: 2.3829 | Valid Accuracy: 32.0682 | Valid Perplexity: 10.84

*New best model at epoch 49*

#### VQ-GAN + Transformer

Train Loss: 1.5007 | Train Accuracy: 53.9200 | Train Perplexity: 4.48

Valid Loss: 1.5057 | Valid Accuracy: 54.1298 | Valid Perplexity: 4.51

*New best model at epoch 49*

#### VQ-GAN (Taming Transformers) + Transformer


Train Loss: 2.1599 | Train Accuracy: 38.3216 | Train Perplexity: 8.67

Valid Loss: 2.1965 | Valid Accuracy: 37.4792 | Valid Perplexity: 8.99

*New best model at epoch 49*

### Split 2

#### VQ-VAE + Transformer

Train Loss: 1.7712 | Train Accuracy: 45.2070 | Train Perplexity: 5.88

Valid Loss: 1.8206 | Valid Accuracy: 43.7583 | Valid Perplexity: 6.18

*New best model at epoch 49*

#### VQ-GAN + Transformer (TO DO)

Train Loss: 2.1945 | Train Accuracy: 37.0628 | Train Perplexity: 8.98

Valid Loss: 2.3233 | Valid Accuracy: 33.9522 | Valid Perplexity: 10.21

*New best model at epoch 49*


#### VQ-GAN (Taming Transformers) + Transformer

### Split 3

#### VQ-VAE + Transformer

Train Loss: 1.0979 | Train Accuracy: 64.5036 | Train Perplexity: 3.00

Valid Loss: 1.1065 | Valid Accuracy: 64.3660 | Valid Perplexity: 3.02

*New best model at epoch 49*

### Split 1 and 2

### VQ-VAE + Transformers
Train Loss: 2.2196 | Train Accuracy: 35.0006 | Train Perplexity: 9.20

Valid Loss: 2.2711 | Valid Accuracy: 33.6416 | Valid Perplexity: 9.69

*New best model at epoch 49*

# Generation

In [ ]:
# VQ-VAE best model
#best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_1/vqvae_best.pth'
#best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_2/vqvae_best.pth'
#best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_3/vqvae_best.pth'
best_save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_1_2/vqvae_best.pth'

checkpoint = torch.load(best_save_path, map_location=torch.device("cpu"), weights_only=False)
transformer.load_state_dict(checkpoint['model_state_dict'])

<All keys matched successfully>

In [ ]:
# VQ-GAN best model
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_1/vqgan_best.pth'
#save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_2/vqgan_best.pth'
#save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_1_2/vqgan_best.pth'
transformer.load_state_dict(torch.load(save_path, weights_only=True))

<All keys matched successfully>

In [ ]:
# VQ-GAN best model for T.T.
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_1/vqgan_tt_best.pth'
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_2/vqgan_tt_best.pth'
save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints/transformers/split_1_2/vqgan_tt_best.pth'
transformer.load_state_dict(torch.load(save_path, map_location=torch.device("cpu"), weights_only=False))

<All keys matched successfully>

In [ ]:
@torch.no_grad()
def generate_and_visualize_textures(transformer, quantizer, device, batch_size=4, seq_len=1024, temperature=0.8, top_k=None, grid_size=(1, 4)):
  transformer.eval()
  quantizer.eval()

  transformer.to(device)
  quantizer.to(device)

  h_lat = w_lat = int(math.sqrt(t))
  embeddings = quantizer.vq.embeddings.weight
  embed_dim = quantizer.vq.embedding_dim
  bos_token_id = transformer.num_embeddings
  sequence = torch.full((batch_size, 1), bos_token_id, dtype=torch.long, device=device)

  with torch.no_grad():
    for _ in range(seq_len):
      logits = transformer(sequence)
      next_token_logits = logits[:, -1, :]                          # (batch_size, num_embeddings)

      next_token_logits /= temperature

      if top_k is not None:
        top_k = min(top_k, next_token_logits.shape[1])
        highest_scores, _ = torch.topk(next_token_logits, top_k, dim=-1)
        threshold = highest_scores[:, -1:]
        next_token_logits[next_token_logits < threshold] = -float('Inf')


      probs = F.softmax(next_token_logits, dim=-1)
      next_token = torch.multinomial(probs, 1)
      sequence = torch.cat([sequence, next_token], dim=1)

    final_indices = sequence[:, 1:]
    z = embeddings[final_indices]
    z = z.view(batch_size, h_lat, w_lat, embed_dim).permute(0, 3, 1, 2).contiguous()
    generated_img = quantizer.decoder(z)
    generated_img = ((generated_img + 1.0) / 2.0).clamp(0, 1) # [-1, 1] -> [0, 1]
    generated_img = generated_img.permute(0, 2, 3, 1).cpu()

    rows, cols = grid_size
    figs, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()

    num_displayed_imgs = min(len(generated_img), len(axes))

    for i in range(num_displayed_imgs):
      axes[i].imshow(generated_img[i])
      axes[i].axis('off')

    plt.show()

In [ ]:
temperatures = [0.5, 0.7, 0.8, 1.0, 1.2]
top_ks = [10, 20, 50, 100]

for temperature in temperatures:
  for top_k in top_ks:
    print(f"Temp: {temperature} | Top-{top_k}")
    #generate_and_visualize_textures(transformer, quantizer, device, temperature=temperature, top_k=top_k, seq_len=256)  # vqgan t.t.
    generate_and_visualize_textures(transformer, quantizer, device, temperature=temperature, top_k=top_k)


# Teacher forcing
A training strategy used in the development of sequence-to-sequence models, by providing the correct input at each step of the sequence rather than allowing the model to generate the next step based on its previous outputs.

In [ ]:
@torch.no_grad()
def teacher_forced_reconstruction(val_loader, transformer, quantizer, device, data, batch_idx):
  transformer.eval()
  quantizer.eval()

  transformer.to(device)
  quantizer.to(device)

  indices = extract_codebook_indices(quantizer, data)
  b, t = indices.shape

  bos_token_id = transformer.num_embeddings
  bos_tokens = torch.full((b, 1), bos_token_id, dtype=torch.long, device=device)
  full_sequence = torch.cat([bos_tokens, indices], dim=1)

  inputs = full_sequence[:, :-1]
  targets = full_sequence[:, 1:]

  logits = transformer(inputs)

  predictions = torch.argmax(logits, dim=-1)

  accuracy = (predictions == targets).float().mean().item() * 100
  logits = logits.reshape(-1, transformer.num_embeddings)
  targets = targets.reshape(-1)
  loss = F.cross_entropy(logits, targets).item()

  h_lat = w_lat = int(math.sqrt(t))
  embeddings = quantizer.vq.embeddings.weight
  embed_dim = quantizer.vq.embedding_dim

  z = embeddings[predictions]
  z = z.view(b, h_lat, w_lat, embed_dim).permute(0, 3, 1, 2).contiguous()
  reconstructed = quantizer.decoder(z)
  reconstructed = (reconstructed + 1.0) / 2.0 # [-1, 1] -> [0, 1]
  reconstructed = reconstructed.permute(0, 2, 3, 1).cpu()

  original = data.cpu()
  original = (original + 1.0) / 2.0
  original = original.permute(0, 2, 3, 1)

  if batch_idx < 10:
    fig, axes = plt.subplots(2, batch_size, figsize=(batch_size * 4, 8))
    for i in range(batch_size):
      axes[0, i].imshow(original[i].clamp(0, 1))
      axes[0, i].set_title(f"Original {i+1}")
      axes[0, i].axis("off")

      axes[1, i].imshow(reconstructed[i].clamp(0, 1))
      axes[1, i].set_title(f"Transformer TF {i+1}")
      axes[1, i].axis("off")
    plt.show()

  return loss, accuracy

In [ ]:
loss, accuracy, perplexity = 0.0, 0.0, 0.0

for batch_idx, (data, _) in enumerate(val_loader):
  data = data.to(device)
  batch_loss, batch_accuracy = teacher_forced_reconstruction(val_loader, transformer, quantizer,device, data, batch_idx)
  loss += batch_loss
  accuracy += batch_accuracy

loss = loss / len(val_loader)
accuracy = accuracy / len(val_loader)
perplexity = np.exp(loss)

print(f"Teacher-forced loss: {loss:.4f}")
print(f"Teacher-forced accuracy: {accuracy:.2f}")
print(f"Teacher-forced perplexity: {perplexity:.2f}")